In [1]:
from src.factor_analysis_v0 import *
from pathlib import Path

data_config = DataConfig(
    underlying_gex_path=r"E:\Pythonfiles\ProjectGamma\results\option_gamma\underlying_gex_daily_with_permno.parquet",
    crsp_daily_path=r"E:\Pythonfiles\ProjectGamma\data\crsp_daily_common_2012_2024_linked_permnos.parquet",
    start_date="2012-01-01",
    end_date="2024-12-31",
    min_abs_price=1.0,
)

column_config = ColumnConfig(
    id_col="permno",
    date_col="date",
    gex_col="net_gex_1pct",
    ret_col="ret",
    spot_col="spot",
    crsp_price_col="prc",
    volume_col="vol",
    control_cols=[
        "price_stock",
        "market_equity_crsp",
        "vol",
        "shrout",
        "spot",
        "total_open_interest",
        "total_option_volume",
    ],
)

preprocess_config = PreprocessConfig(
    winsorize=True,
    winsor_lower=0.01,
    winsor_upper=0.99,
    cross_sectional_zscore=True,
    zscore_suffix="_z",
    min_cross_section_size=60,
    missing_threshold=0.90,
    allow_missing_factors=True,
    use_abs_crsp_price=True,
)

target_config = TargetConfig(
    horizons=[1, 5, 20],
    create_signed_return=True,
    create_abs_return=True,
    create_squared_return=True,
    create_realized_vol=True,
    create_downside_semivar=True,
    create_tail_indicators=True,
    tail_quantiles=[0.05, 0.95],
    tail_groupby="by_date",
    return_type="simple",
)

output_config = OutputConfig(
    output_dir=r"E:\Pythonfiles\ProjectGamma\results\gex_collab_beta",
    save_panel_snapshot=True,
    save_metadata=True,
    panel_snapshot_name="phase1_panel.parquet",
    metadata_name="phase1_metadata.json",
    verbose=True,
)

factor_specs = [
    FactorSpec(
        name="bs_factor_csv",
        path=r"data/factors/BS.csv",
        file_type="csv",
        id_col="PERMNO",
        date_col="date",
        source_id_type="permno",
        frequency="daily",
        factor_cols=[
            "ret", "alpha", "b_mkt", "b_smb", "b_hml", "b_umd",
            "ivol", "tvol", "exret"
        ],
        numeric_only=False,
        suffix="__bs",
        lag_days=0,
    ),
]

identifier_config = IdentifierConfig(
    mapping_path=r"data/secid_crsp_name_mapping_2012_2024.csv",
    mapping_file_type="csv",
    permno_col="permno",
    secid_col="secid",
    ticker_col="ticker",
    cusip_col="cusip",
    ncusip_col="ncusip",
    link_method_col="link_method",
    duplicate_resolution="drop_ambiguous",
)

regression_config = RegressionConfig(
    use_fama_macbeth=True,
    nw_lags=5,
    add_intercept=True,
    use_controls=True,
    control_cols=[
        "price_stock",
        "market_equity_crsp",
        "vol",
        "shrout",
        "spot",
        "total_open_interest",
        "total_option_volume",
    ],
    min_obs_per_date=60,            # was 10
    min_dates_required=60,          # was 20
    regime_validation_y_cols=[
        "ret_fwd_1d",
        "abs_ret_fwd_1d",
        "rv_fwd_5d",
        "tail_left_fwd_1d",
    ],
    interaction_y_cols=[
        "abs_ret_fwd_1d",
        "rv_fwd_5d",
        "tail_left_fwd_1d",
    ],
    n_buckets=5,                    # back to 5 with 231 names
)

classification_config = ClassificationConfig(
    enabled=True,
    target_cols=[
        "tail_left_fwd_1d",
        "tail_left_fwd_5d",
        "tail_abs_fwd_1d",
    ],
    train_end="2020-12-31",
    test_start="2021-01-01",
    test_end="2024-12-31",
    test_size=0.3,
    min_train_rows=3000,            # much larger sample now
    min_test_rows=1000,
    model_names=[
        "logit_l2",
        "logit_elasticnet",
    ],
    max_iter=5000,
    C=0.5,
    l1_ratio=0.3,
    class_weight="balanced",
    use_controls=True,
    control_cols=[
        "price_stock",
        "market_equity_crsp",
        "vol",
        "total_open_interest",
        "total_option_volume",
    ],
    include_factor_interaction=True,
    imputer_strategy="median",
)

phase5_rf_config = RandomForestPhase5Config(
    enabled=True,
    target_cols=[
        "tail_left_fwd_1d",
        "tail_left_fwd_5d",
        "tail_abs_fwd_1d",
    ],
    feature_sets=[
        "factor_only",
        "gex_only",
        "factor_plus_gex",
        "factor_plus_gex_plus_interaction",
    ],
    train_start="2012-01-03",
    train_end="2020-12-31",
    test_start="2021-01-01",
    test_end="2024-12-31",

    selected_factors=[
        "b_mkt__bs",
        "b_smb__bs",
        "b_hml__bs",
        "b_umd__bs",
        "ivol__bs",
        "tvol__bs",
    ],

    n_estimators=500,
    max_depth=5,
    min_samples_leaf=80,
    min_samples_split=160,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,

    min_train_rows=3000,
    min_test_rows=1000,
    min_train_positive=100,
    min_test_positive=40,

    save_scores_csv=True,
    save_importance_csv=True,
    save_predictions_csv=True,

    scores_filename="phase5_rf_scores_beta_2012_2024.csv",
    importance_filename="phase5_rf_feature_importance_beta_2012_2024.csv",
    predictions_filename="phase5_rf_predictions_beta_2012_2024.csv",
)

phase6_overlay_config = Phase6OverlayConfig(
    enabled=True,

    selected_factors=[
        "b_mkt__bs",
        "b_smb__bs",
        "b_hml__bs",
        "b_umd__bs",
        "ivol__bs",
        "tvol__bs",
    ],

    portfolio_return_col="ret_fwd_1d",

    # 231 names -> 5 buckets is now appropriate
    n_buckets=5,
    long_bucket=5,
    short_bucket=1,
    long_short=True,

    # larger universe -> value weight becomes more sensible again
    weighting="value",
    weight_col="market_equity_crsp",

    # ~46 names per tail bucket on average, so this can be much stricter
    min_names_per_side=15,

    run_base=True,
    run_gex_sign_overlay=True,
    run_gex_quantile_overlay=True,
    run_phase5_prob_overlay=True,

    # slightly less defensive than the 20-name setup
    neg_gex_scale=0.75,
    extreme_neg_gex_scale=0.50,

    phase5_target_col="tail_left_fwd_1d",
    phase5_model_name="random_forest",
    phase5_feature_set_preference=[
        "factor_plus_gex_plus_interaction",
        "factor_plus_gex",
        "gex_only",
        "factor_only",
    ],

    # gentler than before; avoid over-cutting exposure
    prob_scale_multiplier=0.50,
    min_scale=0.50,
    max_scale=1.00,

    transaction_cost_bps=5.0,

    save_summary_csv=True,
    save_timeseries_csv=True,
    save_date_scaling_csv=True,

    summary_filename="phase6_overlay_summary_beta_2012_2024.csv",
    timeseries_filename="phase6_overlay_timeseries_beta_2012_2024.csv",
    scaling_filename="phase6_overlay_scaling_by_date_beta_2012_2024.csv",
)

phase7_multivariate_config = Phase7MultivariateConfig(
    enabled=True,
    target_cols=[
        "tail_left_fwd_1d",
        "tail_left_fwd_5d",
        "tail_abs_fwd_1d",
    ],

    # keep compact, but allow a bit more breadth now
    selected_factors=[
        "b_smb__bs",
        "ivol__bs",
        "tvol__bs",
        "b_mkt__bs",
    ],
    include_gex=True,
    include_interactions_with_gex=True,

    train_start="2012-01-03",
    train_end="2020-12-31",
    test_start="2021-01-01",
    test_end="2024-12-31",

    run_logit_elasticnet=True,
    run_hgb=True,

    logit_C=0.5,
    logit_l1_ratio=0.5,
    logit_max_iter=5000,
    logit_class_weight="balanced",

    hgb_learning_rate=0.05,
    hgb_max_iter=300,
    hgb_max_leaf_nodes=31,
    hgb_max_depth=5,
    hgb_min_samples_leaf=100,
    hgb_l2_regularization=0.1,
    hgb_early_stopping=False,

    dropna_for_model=True,
    min_train_rows=3000,
    min_test_rows=1000,
    min_train_positive=100,
    min_test_positive=40,

    compute_permutation_importance=True,
    permutation_n_repeats=10,
    permutation_scoring="roc_auc",

    save_scores_csv=True,
    save_predictions_csv=True,
    save_importance_csv=True,

    scores_filename="phase7_multivariate_scores_beta_2012_2024.csv",
    predictions_filename="phase7_multivariate_predictions_beta_2012_2024.csv",
    importance_filename="phase7_multivariate_permutation_importance_beta_2012_2024.csv",
)


exp = GEXCollaborativeEffectExperiment(
    data_config=data_config,
    column_config=column_config,
    preprocess_config=preprocess_config,
    target_config=target_config,
    regression_config=regression_config,
    classification_config=classification_config,
    output_config=output_config,
    identifier_config=identifier_config,
    phase5_rf_config=phase5_rf_config,
    phase6_overlay_config=phase6_overlay_config,
    phase7_multivariate_config=phase7_multivariate_config,
)

In [2]:
artifacts = exp.run_phase1(
    factor_specs=factor_specs,
    selected_factors=None,
)

panel = artifacts["panel"]
print(panel.head())
print(panel.columns.tolist()[:100])
print(artifacts["metadata"])

    secid  permno       date    spot  spot_close  spot_return  spot_cfadj  \
0  108505   10104 2012-01-03  25.865      25.865     0.008382        13.5   
1  108505   10104 2012-01-04  26.010      26.010     0.005606        13.5   
2  108505   10104 2012-01-05  26.590      26.590     0.022299        13.5   
3  108505   10104 2012-01-06  26.930      26.930     0.012787        13.5   
4  108505   10104 2012-01-09  27.030      27.030     0.005941        13.5   

   spot_shrout  n_contracts  n_optionids  ...  rv_fwd_20d  \
0    5025837.0          241          241  ...    0.011685   
1    5025837.0          247          247  ...    0.011630   
2    5025837.0          242          242  ...    0.010762   
3    5025837.0          234          234  ...    0.010410   
4    5025837.0          241          241  ...    0.010332   

   downside_semivar_fwd_20d  tail_left_fwd_20d  tail_right_fwd_20d  \
0                  0.000019                  0                   0   
1                  0.000019   

In [3]:
artifacts_phase2 = exp.run_phase2(
    factor_cols=None,
)

Phase 2 | Regime validation:   0%|          | 0/4 [00:00<?, ?outcome/s]

E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


Phase 2 | Interaction regression:   0%|          | 0/27 [00:00<?, ?combo/s]

E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Py

In [4]:
artifacts_phase3 = exp.run_phase3(
    factor_cols=None,
    regime_split_y_col="ret_fwd_1d",
    double_sort_y_cols=[
        "ret_fwd_1d",
        "abs_ret_fwd_1d",
        "rv_fwd_5d",
        "tail_left_fwd_1d",
    ],
)

Phase 3 | Regime-split factor test:   0%|          | 0/9 [00:00<?, ?factor/s]

Phase 3 | Double sort:   0%|          | 0/36 [00:00<?, ?combo/s]

In [5]:
artifacts_phase4 = exp.run_phase4(
    factor_cols=None,
    target_cols=[
        "tail_left_fwd_1d",
        "tail_left_fwd_5d",
        "tail_abs_fwd_1d",
    ],
)

Phase 4 | Tail classification:   0%|          | 0/27 [00:00<?, ?combo/s]

In [6]:
phase5_res = exp.run_phase5_random_forest()
phase5_summary = exp.summarize_phase5_random_forest()

print(phase5_summary["top_rows"].head(20))
print(phase5_summary["mean_by_target_feature_set"])
print(phase5_summary["mean_by_factor"].head(20))

Phase 5 | Random Forest:   0%|          | 0/72 [00:00<?, ?combo/s]

          target_col     factor   signal_col  \
40  tail_left_fwd_5d   ivol__bs   ivol__bs_z   
43  tail_left_fwd_5d   ivol__bs   ivol__bs_z   
42  tail_left_fwd_5d   ivol__bs   ivol__bs_z   
44  tail_left_fwd_5d   tvol__bs   tvol__bs_z   
47  tail_left_fwd_5d   tvol__bs   tvol__bs_z   
46  tail_left_fwd_5d   tvol__bs   tvol__bs_z   
64   tail_abs_fwd_1d   ivol__bs   ivol__bs_z   
67   tail_abs_fwd_1d   ivol__bs   ivol__bs_z   
66   tail_abs_fwd_1d   ivol__bs   ivol__bs_z   
16  tail_left_fwd_1d   ivol__bs   ivol__bs_z   
68   tail_abs_fwd_1d   tvol__bs   tvol__bs_z   
19  tail_left_fwd_1d   ivol__bs   ivol__bs_z   
18  tail_left_fwd_1d   ivol__bs   ivol__bs_z   
71   tail_abs_fwd_1d   tvol__bs   tvol__bs_z   
70   tail_abs_fwd_1d   tvol__bs   tvol__bs_z   
20  tail_left_fwd_1d   tvol__bs   tvol__bs_z   
23  tail_left_fwd_1d   tvol__bs   tvol__bs_z   
22  tail_left_fwd_1d   tvol__bs   tvol__bs_z   
28  tail_left_fwd_5d  b_smb__bs  b_smb__bs_z   
31  tail_left_fwd_5d  b_smb__bs  b_smb__

In [3]:
phase6_res = exp.run_phase6_portfolio_overlay()
phase6_summary = exp.summarize_phase6_portfolio_overlay()

print(phase6_summary["mean_by_strategy"])
print(phase6_summary["pivot_sharpe"])
print(phase6_summary["top_rows"].head(20))

Phase 6 | Portfolio overlay:   0%|          | 0/6 [00:00<?, ?factor/s]

Phase 6 | Building daily weights:   0%|          | 0/3144 [00:00<?, ?day/s]

E:\Pythonfiles\ProjectGamma\factor_analysis_v0.py:4673: UserWarning: Skipping phase5 probability overlay for factor 'b_mkt__bs' : Phase 5 random forest results not found in artifacts. Run run_phase5_random_forest() before using phase5 probability overlay.
  warnings.warn(


Phase 6 | Building daily weights:   0%|          | 0/3144 [00:00<?, ?day/s]

E:\Pythonfiles\ProjectGamma\factor_analysis_v0.py:4673: UserWarning: Skipping phase5 probability overlay for factor 'b_smb__bs' : Phase 5 random forest results not found in artifacts. Run run_phase5_random_forest() before using phase5 probability overlay.
  warnings.warn(


...

Phase 6 | Building daily weights:   0%|          | 0/3144 [00:00<?, ?day/s]

E:\Pythonfiles\ProjectGamma\factor_analysis_v0.py:4673: UserWarning: Skipping phase5 probability overlay for factor 'ivol__bs' : Phase 5 random forest results not found in artifacts. Run run_phase5_random_forest() before using phase5 probability overlay.
  warnings.warn(


Phase 6 | Building daily weights:   0%|          | 0/3144 [00:00<?, ?day/s]

E:\Pythonfiles\ProjectGamma\factor_analysis_v0.py:4673: UserWarning: Skipping phase5 probability overlay for factor 'tvol__bs' : Phase 5 random forest results not found in artifacts. Run run_phase5_random_forest() before using phase5 probability overlay.
  warnings.warn(


          strategy_name   ann_ret   ann_vol    sharpe   sortino  max_drawdown  \
1  gex_quantile_overlay  0.098783  0.246468  0.336559  0.482152     -0.621057   
2      gex_sign_overlay  0.098036  0.253274  0.324425  0.462579     -0.645966   
0                  base  0.097298  0.262497  0.309916  0.435402     -0.671063   

   expected_shortfall_5  
1             -0.035836  
2             -0.036907  
0             -0.038557  
strategy_name      base  gex_quantile_overlay  gex_sign_overlay
factor                                                         
b_hml__bs     -0.259703             -0.274869         -0.268230
b_mkt__bs      0.447658              0.469263          0.460311
b_smb__bs      0.383705              0.453728          0.419635
b_umd__bs      0.112377              0.095077          0.104277
ivol__bs       0.587877              0.652973          0.622481
tvol__bs       0.587580              0.623184          0.608076
       factor   signal_col         strategy_name  n_days  m

In [8]:
phase7_res = exp.run_phase7_multivariate_model_training()
phase7_summary = exp.summarize_phase7_multivariate()

print(phase7_summary["mean_by_model"])
print(phase7_summary["best_by_target"])
print(phase7_summary["top_importance"])

E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To 

         target_col              model_name       auc  accuracy  precision  \
0   tail_abs_fwd_1d  hist_gradient_boosting  0.758074  0.895825   0.446363   
1   tail_abs_fwd_1d        logit_elasticnet  0.750544  0.766000   0.247899   
2  tail_left_fwd_1d  hist_gradient_boosting  0.752344  0.948321   0.000000   
3  tail_left_fwd_1d        logit_elasticnet  0.744945  0.764026   0.129738   
4  tail_left_fwd_5d  hist_gradient_boosting  0.766078  0.948534   0.000000   
5  tail_left_fwd_5d        logit_elasticnet  0.763932  0.769031   0.135663   

     recall        f1     brier  
0  0.018625  0.035758  0.083292  
1  0.617668  0.353801  0.195581  
2  0.000000  0.000000  0.046303  
3  0.624781  0.214860  0.201761  
4  0.000000  0.000000  0.045838  
5  0.649352  0.224436  0.199002  
         target_col              model_name  feature_count  \
0   tail_abs_fwd_1d  hist_gradient_boosting              9   
1  tail_left_fwd_1d  hist_gradient_boosting              9   
2  tail_left_fwd_5d  hist_gra